In [1]:
import os
import cv2
from tensorflow.keras.models import load_model
import numpy as np
import torch

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"  

In [ ]:
!pip install PyYAML

In [ ]:
import yaml
data = { 'train' : 'E:\\YOLO8\\화방_데이터\\train',
      'val' : 'E:\\YOLO8\\화방_데이터\\valid',
      'test' : 'E:\\YOLO8\\화방_데이터\\test',
      'names' : ['tom_flower', 'truss','stem'
                 '','','','','','','','','',''
                 ,'','','','','','','','','',''
                 ,'','','','','','','','','',''
                 ,'','','','','','','','','',''
                 ,'','','','','','','','','',''
                 ,'','','','','','','','','',''
                 ,'','','','','','','','','',''
                 ,'','','','','','',''],
      'nc' : 3}

with open('E:\YOLO8\화방_데이터\YOLO8.yaml', 'w') as f:
  yaml.dump(data,f)

with open('E:\YOLO8\화방_데이터\YOLO8.yaml', 'r') as f:
  yolo8_yaml = yaml.safe_load(f)
  display(yolo8_yaml)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')

In [ ]:
print(model)

In [ ]:
import torch.nn as nn
import torch.nn.init as init

def weights_init(m):
    if isinstance(m, nn.Conv2d):
        init.normal_(m.weight, mean=0.0, std=0.02)
    elif isinstance(m, nn.BatchNorm2d):
        init.constant_(m.weight, 1)
        init.constant_(m.bias, 0)

model.apply(weights_init)  # 모든 레이어에 대해 초기화 함수 적용

In [ ]:
import pathlib
from pathlib import Path
pathlib.PosixPath = pathlib.WindowsPath

# 저장된 가중치 파일 로드
loaded_weights = torch.load('YOLO8_trained_weights.pt')

# 모델에 state_dict 로드
model.load_state_dict(loaded_weights)

In [3]:
# 비디오 연결하기
video = cv2.VideoCapture('1')

In [7]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO

model = YOLO('yolov8s.pt', 'YOLO8.yaml')

# 저장된 가중치 파일 로드
loaded_weights = torch.load('YOLO8_trained_weights.pt')

# 모델에 state_dict 로드
model.load_state_dict(loaded_weights)

# 비디오 연결하기
video = cv2.VideoCapture(1)

# 클래스 이름 정의
class_names = {0: 'tom_flower', 1: 'truss', 2: 'stem'}

while video.isOpened():
    ret, frame = video.read()  # 현재 프레임을 읽기
    if not ret:
        break
    
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = np.transpose(img, (2, 0, 1))
    img_tensor = torch.from_numpy(img).unsqueeze(0).float()
    img_tensor /= 255.0

    with torch.no_grad():
        results = model(img_tensor)
    
    # 각 이미지 결과에 대해 반복
    # 클래스별 색상 정의 (BGR 형식)
    class_colors = {
    'tom_flower': (255, 0, 0),  # 빨간색
    'truss': (0, 255, 0),       # 초록색
    'stem': (0, 0, 255)         # 파란색
    }

    for i, result in enumerate(results):
        # 각 검출된 객체에 대해 반복
        for box, cls in zip(result.boxes.xyxy, result.boxes.cls):
            # CUDA 텐서를 CPU로 이동한 후 NumPy 배열로 변환
            x1, y1, x2, y2 = box.cpu().numpy()
            class_index = int(cls.cpu())  # 클래스 인덱스도 CPU로 이동이 필요할 수 있습니다.
            class_name = class_names.get(class_index, 'Unknown')  # 클래스 이름
            color = class_colors.get(class_name, (255, 255, 255))  # 클래스에 해당하는 색상을 가져오거나, 정의되지 않았다면 흰색 사용
        
            # 바운딩 박스 그리기
            start_point = (int(x1), int(y1))
            end_point = (int(x2), int(y2))
            thickness = 2
            cv2.rectangle(frame, start_point, end_point, color, thickness)
        
            # 클래스 이름과 신뢰도 표시
            label = f'{class_name}'
            cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


    cv2.imshow('frame', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 20.0ms
Speed: 1.0ms preprocess, 20.0ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 21.0ms
Speed: 1.0ms preprocess, 21.0ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 21.0ms
Speed: 1.0ms preprocess, 21.0ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 22.0ms
Speed: 2.0ms preprocess, 22.0ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 20.0ms
Speed: 1.0ms preprocess, 20.0ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 19.0ms
Speed: 1.0ms preprocess, 19.0ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 20.0ms
Speed: 1.0ms preprocess, 20.0ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 bicycle, 19.0ms
Speed: 1.0ms preprocess, 19.

In [ ]:
# 클래스별 색상 정의 (BGR 형식)
# 신뢰도를 추가하고 싶은경우 아래 코드로 교체
class_colors = {
    'tom_flower': (255, 0, 0),  # 빨간색
    'truss': (0, 255, 0),       # 초록색
    'stem': (0, 0, 255)         # 파란색
}

for i, result in enumerate(results):
    # 각 검출된 객체에 대해 반복
    for box, cls, score in zip(result.boxes.xyxy, result.boxes.cls, result.boxes.scores):
        # CUDA 텐서를 CPU로 이동한 후 NumPy 배열로 변환
        x1, y1, x2, y2 = box.cpu().numpy()
        class_index = int(cls.cpu())  # 클래스 인덱스도 CPU로 이동이 필요할 수 있습니다.
        class_name = class_names.get(class_index, 'Unknown')  # 클래스 이름
        color = class_colors.get(class_name, (255, 255, 255))  # 클래스에 해당하는 색상을 가져오거나, 정의되지 않았다면 흰색 사용
        confidence = score.cpu().item()  # 신뢰도
        
        # 바운딩 박스 그리기
        start_point = (int(x1), int(y1))
        end_point = (int(x2), int(y2))
        thickness = 2
        cv2.rectangle(frame, start_point, end_point, color, thickness)
        
        # 클래스 이름과 신뢰도 표시
        label = f'{class_name}: {confidence:.2f}'
        cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
